# E-Commerce Sentiment Analysis
**TF-IDF + Logistic Regression on Women's E-Commerce Clothing Reviews**

This notebook drives the code that already lives in:
- `app/training/train_logistic_regression.py` — data loading, pipeline, training, saving
- `app/resources/api.py` — FastAPI app with the `/analyse` endpoint

Run from the **project root** so imports resolve correctly.

In [ ]:
import sys
import inspect
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# ── import from our own app modules ──────────────────────────────────────────
from app.training.train_logistic_regression import (
    load_and_prepare_data,
    train_model,
    DATA_PATH,
    MODEL_PATH,
    TEXT_COL,
    RATING_COL,
)

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110

SEED      = 42
LABEL_COL = "label"
CLASS_LABELS = {0: "negative", 1: "positive"}

print(f"Data  : {DATA_PATH}")
print(f"Model : {MODEL_PATH}")

---
## 1. Dataset Overview

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Shape   : {df_raw.shape}")
print(f"Columns : {df_raw.columns.tolist()}")
df_raw.head(3)

In [ ]:
print("=== Missing values ===")
print(df_raw.isnull().sum())
print("\n=== Dtypes ===")
print(df_raw.dtypes)

In [ ]:
df_raw["review_length"] = df_raw[TEXT_COL].fillna("").str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

rating_counts = df_raw[RATING_COL].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values, color="steelblue", edgecolor="white")
axes[0].set_title("Rating Distribution (all reviews)")
axes[0].set_xlabel("Star Rating")
axes[0].set_ylabel("Count")
for i, v in zip(rating_counts.index, rating_counts.values):
    axes[0].text(i, v + 50, str(v), ha="center", fontsize=9)

axes[1].hist(df_raw["review_length"], bins=50, color="steelblue", edgecolor="white")
axes[1].set_title("Review Length Distribution (words)")
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()
print(f"Avg review length: {df_raw['review_length'].mean():.1f} words | Median: {df_raw['review_length'].median():.0f}")

---
## 2. Label Creation

`load_and_prepare_data()` in `train_logistic_regression.py` handles this:
- Rating 3 → dropped (neutral)
- Ratings 1–2 → **negative (0)**
- Ratings 4–5 → **positive (1)**

In [ ]:
# Show the actual source from our training module
print(inspect.getsource(load_and_prepare_data))

In [ ]:
X, y = load_and_prepare_data()

print(f"Total samples after filtering: {len(X):,}")
print(f"Positive (1): {y.sum():,}  ({y.mean()*100:.1f}%)")
print(f"Negative (0): {(y==0).sum():,}  ({(y==0).mean()*100:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Negative", "Positive"], [(y==0).sum(), y.sum()],
       color=["#e74c3c", "#2ecc71"], edgecolor="white")
ax.set_title("Class Distribution after Label Creation")
ax.set_ylabel("Count")
for i, v in enumerate([(y==0).sum(), y.sum()]):
    ax.text(i, v + 30, str(v), ha="center")
plt.tight_layout()
plt.show()

---
## 3. Data Cleaning

`load_and_prepare_data()` drops rows where `Review Text` or `Rating` is null and removes neutral reviews. Below we inspect what the cleaned text looks like.

In [ ]:
# Compare raw vs cleaned counts
raw_count    = len(df_raw)
no_null      = df_raw[[TEXT_COL, RATING_COL]].dropna()
no_neutral   = no_null[no_null[RATING_COL] != 3]

print(f"Raw rows         : {raw_count:,}")
print(f"After dropna     : {len(no_null):,}  (-{raw_count - len(no_null):,})")
print(f"After drop 3★    : {len(no_neutral):,}  (-{len(no_null) - len(no_neutral):,})")
print(f"Final (X)        : {len(X):,}")

In [ ]:
df_clean = X.to_frame().copy()
df_clean[LABEL_COL] = y.values

print("=== Sample negative reviews ===")
for txt in df_clean[df_clean[LABEL_COL] == 0][TEXT_COL].sample(3, random_state=SEED).values:
    print(f"  • {txt[:130]}\n")

print("=== Sample positive reviews ===")
for txt in df_clean[df_clean[LABEL_COL] == 1][TEXT_COL].sample(3, random_state=SEED).values:
    print(f"  • {txt[:130]}\n")

---
## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Train : {len(X_train):,} reviews")
print(f"Test  : {len(X_test):,} reviews")
print(f"\nTrain — negative: {(y_train==0).sum():,}  positive: {(y_train==1).sum():,}")
print(f"Test  — negative: {(y_test==0).sum():,}  positive: {(y_test==1).sum():,}")

---
## 5. TF-IDF Feature Extraction

The vectoriser settings come from `train_logistic_regression.py`. We fit on the training split only to prevent data leakage.

In [ ]:
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.90,
    max_features=50_000,
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"Vocabulary size : {len(tfidf.vocabulary_):,} tokens")
print(f"Train matrix    : {X_train_tfidf.shape}")
print(f"Test matrix     : {X_test_tfidf.shape}")
print(f"Sparsity        : {1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]):.4%}")

In [ ]:
doc_freq   = np.diff(X_train_tfidf.tocsc().indptr)
feat_names = tfidf.get_feature_names_out()
top_idx    = np.argsort(doc_freq)[-20:][::-1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh([feat_names[i] for i in top_idx], [doc_freq[i] for i in top_idx],
        color="steelblue", edgecolor="white")
ax.set_title("Top 20 Most Frequent TF-IDF Tokens (by document count)")
ax.set_xlabel("Documents")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## 6. Logistic Regression Model

We call `train_model()` from `train_logistic_regression.py` directly — it builds the full `TfidfVectorizer → LogisticRegression` pipeline, prints metrics, and saves the model to `app/model/`.

In [ ]:
print(inspect.getsource(train_model))

In [ ]:
train_model()  # trains, prints eval, saves model to MODEL_PATH

In [ ]:
# Load the saved model — same object that api.py uses at startup
model = joblib.load(MODEL_PATH)

clf_step   = model.named_steps["classifier"]
tfidf_step = model.named_steps["tfidf"]
coefs      = clf_step.coef_[0]
names      = tfidf_step.get_feature_names_out()

n = 15
top_pos = np.argsort(coefs)[-n:][::-1]
top_neg = np.argsort(coefs)[:n]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh([names[i] for i in top_pos], [coefs[i] for i in top_pos],
         color="#2ecc71", edgecolor="white")
ax1.set_title(f"Top {n} Positive Features")
ax1.set_xlabel("Coefficient")
ax1.invert_yaxis()

ax2.barh([names[i] for i in top_neg], [coefs[i] for i in top_neg],
         color="#e74c3c", edgecolor="white")
ax2.set_title(f"Top {n} Negative Features")
ax2.set_xlabel("Coefficient")
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

---
## 7. Evaluation

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.4f}  ({acc*100:.2f}%)\n")
print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))

In [ ]:
cv_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy", n_jobs=-1)
print(f"5-Fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Fold scores: {[round(s, 4) for s in cv_scores]}")

---
## 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay(cm, display_labels=["Negative", "Positive"]).plot(
    ax=axes[0], colorbar=False, cmap="Blues"
)
axes[0].set_title("Confusion Matrix — Counts")

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cm_norm, display_labels=["Negative", "Positive"]).plot(
    ax=axes[1], colorbar=False, cmap="Blues", values_format=".2%"
)
axes[1].set_title("Confusion Matrix — Row-Normalised")

plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True  Negatives : {tn:,}")
print(f"False Positives : {fp:,}  ← negative reviews labelled positive")
print(f"False Negatives : {fn:,}  ← positive reviews labelled negative")
print(f"True  Positives : {tp:,}")

---
## 9. Error Analysis

In [ ]:
errors = X_test.copy().to_frame()
errors["true"]       = y_test.values
errors["predicted"]  = y_pred
errors["confidence"] = y_proba.max(axis=1)
errors = errors[errors["true"] != errors["predicted"]].sort_values("confidence", ascending=False)

print(f"Misclassified: {len(errors):,} / {len(y_test):,}  ({len(errors)/len(y_test)*100:.1f}%)\n")

print("=== High-confidence False Positives (negative → called positive) ===")
for _, row in errors[errors["true"] == 0].head(4).iterrows():
    print(f"  [conf={row['confidence']:.2f}] {row[TEXT_COL][:140]}\n")

print("=== High-confidence False Negatives (positive → called negative) ===")
for _, row in errors[errors["true"] == 1].head(4).iterrows():
    print(f"  [conf={row['confidence']:.2f}] {row[TEXT_COL][:140]}\n")

In [ ]:
def explain_prediction(text, pipeline):
    """Per-token contribution breakdown for a single review."""
    tfidf_s = pipeline.named_steps["tfidf"]
    clf_s   = pipeline.named_steps["classifier"]
    coefs_  = clf_s.coef_[0]
    names_  = tfidf_s.get_feature_names_out()

    vec  = tfidf_s.transform([text])
    nz   = vec.nonzero()[1]
    pred = clf_s.predict(vec)[0]
    prob = clf_s.predict_proba(vec)[0]

    contribs = sorted(
        [(names_[i], coefs_[i], float(vec[0, i])) for i in nz],
        key=lambda x: x[1] * x[2]
    )
    print(f"Prediction : {CLASS_LABELS[pred]}  (neg={prob[0]:.3f}  pos={prob[1]:.3f})")
    print(f"{'Feature':<28} {'Coef':>8} {'TF-IDF':>8} {'Contribution':>13}")
    print("-" * 61)
    for feat, coef, val in contribs:
        print(f"{feat:<28} {coef:>8.4f} {val:>8.4f} {coef*val:>13.4f}")
    net = sum(c * v for _, c, v in contribs) + clf_s.intercept_[0]
    print(f"{'Net score':<28} {'':>17} {net:>13.4f}  (>0 = positive)")

fn_df = errors[errors["true"] == 1]
if len(fn_df):
    example = fn_df.iloc[0][TEXT_COL]
    print(f"Review: {example[:200]}\n")
    explain_prediction(example, model)

---
## 10. Model Improvement

Compare regularisation strength (`C`) and n-gram range without changing `train_logistic_regression.py`.

In [ ]:
experiments = [
    {"C": 0.1,  "ngram_range": (1, 1), "label": "C=0.1  unigrams"},
    {"C": 0.3,  "ngram_range": (1, 2), "label": "C=0.3  bigrams  ← current"},
    {"C": 1.0,  "ngram_range": (1, 2), "label": "C=1.0  bigrams"},
    {"C": 5.0,  "ngram_range": (1, 2), "label": "C=5.0  bigrams"},
    {"C": 0.3,  "ngram_range": (1, 3), "label": "C=0.3  trigrams"},
]

results = []
for exp in experiments:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True, stop_words="english",
            ngram_range=exp["ngram_range"],
            min_df=3, max_df=0.90, max_features=50_000,
        )),
        ("clf", LogisticRegression(
            C=exp["C"], max_iter=1000,
            class_weight="balanced", random_state=SEED,
        )),
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    results.append({
        "Config"        : exp["label"],
        "Accuracy"      : round(accuracy_score(y_test, preds), 4),
        "F1 (negative)" : round(f1_score(y_test, preds, pos_label=0), 4),
        "F1 (positive)" : round(f1_score(y_test, preds, pos_label=1), 4),
        "F1 (macro)"    : round(f1_score(y_test, preds, average="macro"), 4),
    })

results_df = pd.DataFrame(results).set_index("Config")
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
x, w = np.arange(len(results_df)), 0.2
ax.bar(x - w,   results_df["Accuracy"],      width=w, label="Accuracy",     color="steelblue")
ax.bar(x,       results_df["F1 (negative)"], width=w, label="F1 negative",  color="#e74c3c")
ax.bar(x + w,   results_df["F1 (positive)"], width=w, label="F1 positive",  color="#2ecc71")
ax.bar(x + 2*w, results_df["F1 (macro)"],   width=w, label="F1 macro",     color="#9b59b6")
ax.set_xticks(x + w / 2)
ax.set_xticklabels(results_df.index, rotation=15, ha="right", fontsize=9)
ax.set_ylim(0.5, 1.0)
ax.set_title("Model Comparison across Configurations")
ax.legend()
plt.tight_layout()
plt.show()

---
## 11. Save Model

`train_model()` already handles saving. Here we verify the artefact on disk and smoke-test it.

In [ ]:
size_kb = MODEL_PATH.stat().st_size / 1024
print(f"Model path : {MODEL_PATH}")
print(f"File size  : {size_kb:.1f} KB")

loaded = joblib.load(MODEL_PATH)
test_reviews = [
    "Absolutely love this dress, fits perfectly and the material is so soft!",
    "Terrible quality. Fell apart after the first wash. Complete waste of money.",
    "Nice colour but runs very small. Had to return it.",
    "Great value for the price, looks even better in person!",
]
preds  = loaded.predict(test_reviews)
probas = loaded.predict_proba(test_reviews)

print(f"\n{'Review':<65} {'Sentiment':<10} {'Confidence'}")
print("-" * 90)
for txt, p, pr in zip(test_reviews, preds, probas):
    print(f"{txt[:63]:<65} {CLASS_LABELS[p]:<10} {pr[p]:.4f}")

---
## 12. FastAPI Demo

We import the real `app` object from `app/resources/api.py` and call it through FastAPI's `TestClient` — no server process needed.

In [ ]:
# Show the actual endpoint source from api.py
from app.resources import api as api_module
print(inspect.getsource(api_module))

In [ ]:
from fastapi.testclient import TestClient
from app.resources.api import app as fastapi_app

client = TestClient(fastapi_app)

# Health check
r = client.get("/health")
print("GET /health →", r.json())

In [ ]:
demo_reviews = [
    {
        "comment": "Absolutely love this dress, fits perfectly and the material is so soft!",
        "product_name": "Floral Summer Dress",
        "category": "clothing",
        "rating": 5.0,
        "verified_purchase": True,
    },
    {
        "comment": "Terrible quality. Fell apart after the first wash. Complete waste of money.",
        "product_name": "Casual Blouse",
        "category": "clothing",
        "rating": 1.0,
        "verified_purchase": True,
    },
    {
        "comment": "Nice colour but runs very small. Had to return it.",
        "product_name": "Slim Fit Jeans",
        "category": "clothing",
        "rating": 2.0,
    },
    {
        "comment": "Great value for the price, looks even better in person!",
        "product_name": "Printed Maxi Skirt",
        "category": "clothing",
        "rating": 4.0,
        "verified_purchase": True,
    },
]

print(f"POST /analyse\n{'Comment':<62} {'Sentiment':<10} {'Confidence'}")
print("-" * 87)
for review in demo_reviews:
    resp = client.post("/analyse", json=review)
    resp.raise_for_status()
    data = resp.json()
    print(f"{data['comment_preview'][:60]:<62} {data['sentiment']:<10} {data['confidence']:.4f}")

In [ ]:
# Full JSON response for a single review
resp = client.post("/analyse", json={
    "comment": "The fabric feels cheap but it looks great on and the price is fair.",
    "product_name": "Cotton Blouse",
    "rating": 3.5,
})
import json
print(json.dumps(resp.json(), indent=2))